In [31]:
from gurobipy import Model, GRB

# Constants (Example values, adjust as needed)
max_time = 200  # Max number of rows
num_tasks = 5   # Number of tasks
task_times = {j: 2 for j in range(J)}  # All tasks take 2 time blocks
deadlines = {j: 100 for j in range(J)}  # Deadline is 100 for each block

print(f"{task_times}, {d}")

# Create a new model
model = Model("PSC-MIP-V2")

# Decision Variables
task_to_block = model.addVars(max_time, num_tasks, vtype=GRB.BINARY, name="x")  # x[i, j] binary
task_starting_time = model.addVars(max_time, num_tasks, vtype=GRB.BINARY, name="t")  # t[i, j] binary

# Constraints
"""
"""
for j in range(num_tasks):
    # Constraint 1: Block assignment
    # for i in range(d[j]):
    #     for k in range(b[j]):
    #         if i - k >= 0:
    #             model.addConstr(x[i, j] >= t[i - k, j], f"Block_{i}_{k}_{j}")


    # Constraint 2: Total assignment constraint
    for j in range(num_tasks):
        model.addConstr(sum(task_to_block[i, j] for i in range(deadlines[j])) == task_times[j], f"Sum_x_{j}")

    # Constraint 3: Bound constraint on t_{ij}
    for i in range(max_time):
        model.addConstr(i*task_starting_time[i, j] <= (deadlines[j] - task_times[j]), f"Bound_t_{i}_{j}")

    # Constraint 4: Unique start position constraint
    for j in range(num_tasks):
        model.addConstr(sum(task_starting_time[i, j] for i in range(max_time)) == 1, f"Unique_t_{j}")

    # Constraint 5: 
    # for i in range(max_time):
    #     model.addConstr(sum(task_to_block[i, j] for j in range(num_tasks)) == 1, f"One task per time block")

# Set objective (optional, if you want an optimization goal)
model.setObjective(sum(task_to_block[i, j] for i in range(max_time) for j in range(num_tasks)), GRB.MAXIMIZE)

# Solve the model
a = model.optimize()

print(f"TYPE OF MODEL: {type(a)}")


# Print results
if model.status == GRB.OPTIMAL:
    print("Optimal Solution Found:")
    for j in range(num_tasks):
        for i in range(max_time):
            if task_to_block[i, j].x > 0.5:
                print(f"task_to_block[{i},{j}] = {task_to_block[i, j].x}")
            if task_starting_time[i, j].x > 0.5:
                print(f"task_starting_time[{i},{j}] = {task_starting_time[i, j].x}")
else:
    print("No optimal solution found.")





{0: 2, 1: 2, 2: 2, 3: 2, 4: 2}, {0: 100, 1: 100, 2: 100, 3: 100, 4: 100}
Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[arm] - Darwin 23.2.0 23C71)

CPU model: Apple M1
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Academic license 2548635 - for non-commercial use only - registered to ba___@rice.edu
Optimize a model with 1050 rows, 2000 columns and 8495 nonzeros
Model fingerprint: 0x15edf945
Variable types: 0 continuous, 2000 integer (2000 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+02]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+02]
Found heuristic solution: objective 510.0000000
Presolve removed 1050 rows and 2000 columns
Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 1 (of 8 available processors)

Solution count 1: 510 

Optimal solution found (tolerance 1.00e-04)
Best 

In [ ]:
# Class for PSC Optimizer
class PSC_Optimizer:
    
    def __init__(self, task_times, deadlines, task_names):
        self.max_time = max(deadline for deadline in deadlines)
        self.task_times = task_times # Array (length num tasks)
        self.deadlines = deadlines # Array (length num tasks)
        self.task_names = task_names
        self.num_tasks = len(task_times) - 1
        
    
    def OptimizeCalendar(self):
        if (len(self.task_times) != len(self.deadlines)):
            raise Exception

        # Define dicts for task times and deadlines
        task_times = {j: self.task_times[j] for j in range(self.num_tasks)}  # All tasks take 2 time blocks
        deadlines = {j: self.deadlines[j] for j in range(self.num_tasks)}  # Deadline is 100 for each block

        # Create a new model
        model = Model("PSC-MIP-V2")

        # Decision Variables
        task_to_block = model.addVars(max_time, num_tasks, vtype=GRB.BINARY, name="x")  # x[i, j] binary
        task_starting_time = model.addVars(max_time, num_tasks, vtype=GRB.BINARY, name="t")  # t[i, j] binary

        # Constraints
        print(f"Number of tasks: {num_tasks}")
        for j in range(num_tasks):
            # Constraint 1: Block assignment
            # for i in range(d[j]):
            #     for k in range(b[j]):
            #         if i - k >= 0:
            #             model.addConstr(x[i, j] >= t[i - k, j], f"Block_{i}_{k}_{j}")


            # Constraint 2: Total assignment constraint
            for j in range(num_tasks):
                print(f"j: {j}")
                model.addConstr(sum(task_to_block[i, j] for i in range(self.deadlines[j])) == self.task_times[j], f"Sum_x_{j}")
                print(j, ": ", self.deadlines[j])

            # Constraint 3: Bound constraint on t_{ij}
            for i in range(max_time):
                model.addConstr(i*self.task_starting_time[i, j] <= (self.deadlines[j] - self.task_times[j]), f"Bound_t_{i}_{j}")

            # Constraint 4: Unique start position constraint
            for j in range(num_tasks):
                model.addConstr(sum(self.task_starting_time[i, j] for i in range(self.max_time)) == 1, f"Unique_t_{j}")

            # Constraint 5: 
            for i in range(self.max_time):
                model.addConstr(sum(task_to_block[i, j] for j in range(self.num_tasks)) == 1, f"One task per time block")

        # Set objective (optional, if you want an optimization goal)
        model.setObjective(sum(task_to_block[i, j] for i in range(self.max_time) for j in range(self.num_tasks)), GRB.MAXIMIZE)

        # Solve the model
        a = model.optimize()

        # Post-processing
        # Return type: {task: (start_time, end_time)}
        
        # Store results
        results_dict = dict()
        if model.status == GRB.OPTIMAL:
            print("Optimal Solution Found:")
            for j in range(num_tasks):
                for i in range(max_time):
                    if task_starting_time[i, j].x > 0.5:
                        start_time = task_starting_time[i, j].x
                        end_time = task_starting_time[i, j].x + self.task_times[j]
                        results_dict[j] = tuple(start_time, end_time)
        else:
            print("No optimal solution found.")

        return results_dict
    

In [52]:
# Instance Test 1
task_times_1 = [1, 1, 2]
deadlines_1 = [10, 12, 30]
task_names_1 = ["task1", "task2", "task3"]
    
# Create instance
cal_1 = PSC_Optimizer(task_times_1, deadlines_1, task_names_1)

# Optimize
print(cal_1.OptimizeCalendar())


j: 0
0 :  10
j: 1
1 :  12
j: 2
2 :  30
j: 3


IndexError: list index out of range